# 13 - DINOv2 ViT-L モデル評価

## 概要
Meta AI の DINOv2 ViT-L モデルを評価する。

## モデル情報
- **Model**: facebook/dinov2-large
- **Embedding dimension**: 1024
- **Type**: Image only (テキスト埋め込み非対応)
- **特徴**: 教師なし学習による強力な視覚特徴抽出

In [1]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

from image_vector_poc import DINOv2Embedder
from image_vector_poc.evaluation import EvaluationReporter, evaluate_embeddings

## 設定

In [2]:
DB_PATH = Path("../data/images.duckdb")
OUTPUT_DIR = Path("../data/evaluations")
BATCH_SIZE = 32
RANDOM_STATE = 42

## 画像カタログの読み込み

In [3]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
query = """
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY category, file_name
"""
catalog = conn.execute(query).fetchall()
conn.close()

image_ids = [r[0] for r in catalog]
file_paths = [r[1] for r in catalog]
categories = [r[2] for r in catalog]

category_labels = np.array(categories)
category_counts = pd.Series(categories).value_counts().to_dict()
unique_categories = list(category_counts.keys())

print(f"Total images: {len(catalog)}")
print(f"Categories: {unique_categories}")

Total images: 378
Categories: ['EuroPython2025', 'PyConJP2025', 'PyConJP2025-PreCampHiroshima', 'KashiwaVillagePark2026', 'TokyoNight202505', 'terada']


## モデルの初期化

In [4]:
print("Loading DINOv2 ViT-L model...")
embedder = DINOv2Embedder(device="cuda")
print(f"Model: {embedder.model_name}")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"Device: {embedder.device}")

Loading DINOv2 ViT-L model...


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Model: facebook/dinov2-large
Embedding dimension: 1024
Device: cuda


## 画像のベクトル化

In [5]:
print(f"Generating embeddings for {len(file_paths)} images...")

start_time = time.time()
embeddings_list = []

for i in tqdm(range(0, len(file_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = file_paths[i:i + BATCH_SIZE]
    batch_images = []
    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            batch_images.append(Image.new("RGB", (224, 224), color="gray"))
    
    batch_embs = embedder.embed_images(batch_images)
    embeddings_list.append(batch_embs)

embeddings = np.vstack(embeddings_list)
processing_time = time.time() - start_time

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Processing time: {processing_time:.2f}s")
print(f"Speed: {len(file_paths) / processing_time:.1f} images/sec")

Generating embeddings for 378 images...


Embedding:   0%|          | 0/12 [00:00<?, ?it/s]


Embeddings shape: (378, 1024)
Processing time: 94.63s
Speed: 4.0 images/sec


## 評価の実行

In [6]:
print("Running evaluation...\n")

metrics = evaluate_embeddings(
    embeddings=embeddings,
    labels=category_labels,
    model_name=embedder.model_name,
    embedding_dim=embedder.embedding_dim,
    categories=unique_categories,
    category_counts=category_counts,
    processing_time=processing_time,
    random_state=RANDOM_STATE,
)

Running evaluation...



## 結果の表示

In [7]:
print("=" * 60)
print("Evaluation Results - DINOv2 ViT-L")
print("=" * 60)
print(f"Model: {metrics.model_name}")
print(f"Embedding dimension: {metrics.embedding_dim}")
print(f"Processing time: {metrics.processing_time_seconds:.2f}s")

print("\n--- t-SNE Metrics ---")
print(f"2D: Silhouette={metrics.silhouette_2d:.4f}, Trust={metrics.trustworthiness_2d:.4f}, DistRatio={metrics.distance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.silhouette_3d:.4f}, Trust={metrics.trustworthiness_3d:.4f}, DistRatio={metrics.distance_ratio_3d:.4f}")

print("\n--- PCA Metrics ---")
print(f"2D: Silhouette={metrics.pca_silhouette_2d:.4f}, Variance={metrics.pca_variance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.pca_silhouette_3d:.4f}, Variance={metrics.pca_variance_ratio_3d:.4f}")

Evaluation Results - DINOv2 ViT-L
Model: facebook/dinov2-large
Embedding dimension: 1024
Processing time: 94.63s

--- t-SNE Metrics ---
2D: Silhouette=0.0835, Trust=0.9540, DistRatio=0.5577
3D: Silhouette=0.0364, Trust=0.9558, DistRatio=0.6764

--- PCA Metrics ---
2D: Silhouette=-0.0050, Variance=0.1733
3D: Silhouette=0.0079, Variance=0.2243


## 結果の保存

In [8]:
reporter = EvaluationReporter(OUTPUT_DIR)
filepath = reporter.save(metrics)
print(f"Results saved to: {filepath}")

Results saved to: ../data/evaluations/facebook_dinov2-large_2026-02-03.json


## GPUメモリのクリーンアップ

In [9]:
del embedder
del embeddings

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

GPU memory cleared.
